<a href="https://colab.research.google.com/github/thiselvan/ml-project/blob/main/Intermediate_SQL_Querying.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import sqlite3

customers_url = "https://raw.githubusercontent.com/graphql-compose/graphql-compose-examples/master/examples/northwind/data/csv/customers.csv"
orders_url = "https://raw.githubusercontent.com/graphql-compose/graphql-compose-examples/master/examples/northwind/data/csv/orders.csv"

customers_df = pd.read_csv(customers_url)
orders_df = pd.read_csv(orders_url)

conn = sqlite3.connect(":memory:")
customers_df.to_sql("customers", conn, index=False, if_exists="replace")
orders_df.to_sql("orders", conn, index=False, if_exists="replace")


830

Relevant columns:

customers: CustomerID, CompanyName, Country
orders: OrderID, CustomerID, Freight (treat Freight as the order amount)


Task 1 — Aggregation and Grouping

Using only the orders table, write a SQL query that returns each CustomerID along with:

The total number of orders they placed (order_count)
The total freight amount across all their orders (total_freight)
The average freight amount per order (avg_freight)
Sort the results by total_freight in descending order.

Run the query using pd.read_sql_query() and display the top 10 rows.



In [ ]:
pd.read_sql_query("\
SELECT CustomerId , COUNT(OrderId) as order_count, SUM(Freight) as total_freight, AVG(Freight) as avg_freight \
FROM  orders \
GROUP BY CustomerId \
ORDER BY total_freight DESC \
LIMIT 10; \
",conn)

,customerID,order_count,total_freight,avg_freight
0,SAVEA,31,6683.70,215.603226
1,ERNSH,30,6205.39,206.846333
2,QUICK,28,5605.63,200.201071
3,HUNGO,19,2755.24,145.012632
4,RATTC,18,2134.21,118.567222
5,QUEEN,13,1982.70,152.515385
6,FOLKO,19,1678.08,88.320000
7,BERGS,18,1559.52,86.640000
8,FRANK,15,1403.44,93.562667
9,MEREP,13,1394.22,107.247692


Task 2 — WHERE vs. HAVING

Write two separate SQL queries to demonstrate the difference between WHERE and HAVING:

Query A: From the orders table, filter rows where Freight is greater than 50 before aggregation, then group by CustomerID and return the count of such orders as high_freight_orders.



In [ ]:
pd.read_sql_query("\
SELECT CustomerId, COUNT(OrderId) AS high_freight_orders\
 FROM ORDERS\
 WHERE Freight > 50\
 GROUP BY CustomerId; \
",conn)

,customerID,high_freight_orders
0,ALFKI,2
1,ANTON,2
2,AROUT,2
3,BERGS,11
4,BLAUS,1
...,...,...
69,WANDK,2
70,WARTH,6
71,WELLI,1
72,WHITC,7


Query B: From the orders table, group by CustomerID and return only those customers whose total freight exceeds 500, using HAVING. Return CustomerID and total_freight.

In a markdown cell below your queries, write 2–3 sentences explaining why Query A and Query B produce different results even though both involve a threshold on Freight.



In [ ]:
pd.read_sql_query(" \
SELECT CustomerId, sum(Freight) AS total_freight \
FROM Orders \
GROUP BY CustomerId HAVING total_freight>50;\
",conn)

,customerID,total_freight
0,ALFKI,225.58
1,ANATR,97.42
2,ANTON,268.52
3,AROUT,471.95
4,BERGS,1559.52
...,...,...
79,WARTH,822.48
80,WELLI,194.71
81,WHITC,1353.06
82,WILMK,88.41


Query A applies the WHERE clause before grouping, so only individual orders with Freight > 50 are considered, and then those filtered rows are counted per customer. Query B, on the other hand, aggregates all orders per customer first, then uses HAVING to filter customers whose total freight exceeds 500. The difference lies in the stage of filtering: WHERE filters rows before aggregation, while HAVING filters groups after aggregation.

Task 3 — JOIN and Aggregation

Write a SQL query that joins the customers and orders tables on CustomerID and returns:

CompanyName (from customers)
Country (from customers)
Total number of orders placed (order_count)
Total freight (total_freight)
Include only customers who have placed at least one order (i.e., use INNER JOIN).



In [ ]:
pd.read_sql_query("\
SELECT c.CompanyName, c.Country, COUNT(o.Freight) AS order_count, SUM(o.Freight) AS total_freight \
FROM Customers as c INNER JOIN Orders as o \
ON c.CustomerId = o.CustomerId \
GROUP BY c.CompanyName,c.Country;\
",conn)

,companyName,country,order_count,total_freight
0,Alfreds Futterkiste,Germany,6,225.58
1,Ana Trujillo Emparedados y helados,Mexico,4,97.42
2,Antonio Moreno Taquería,Mexico,7,268.52
3,Around the Horn,UK,13,471.95
4,B's Beverages,UK,10,281.31
...,...,...,...,...
84,Wartian Herkku,Finland,15,822.48
85,Wellington Importadora,Brazil,9,194.71
86,White Clover Markets,USA,14,1353.06
87,Wilman Kala,Finland,7,88.41


Then write a second query using LEFT JOIN that includes all customers, even those with no orders. For customers with no orders, total_freight should appear as NULL or 0.

Display both results and in a markdown cell, explain in 2–3 sentences what changed between the two queries and why.



In [ ]:
pd.read_sql_query("\
SELECT c.CompanyName, c.Country, COUNT(o.Freight) AS order_count, \
CASE WHEN SUM(o.Freight) IS NULL THEN 0 ELSE SUM(o.Freight) END AS total_freight \
FROM Customers as c LEFT JOIN Orders as o \
ON c.CustomerId = o.CustomerId \
GROUP BY c.CompanyName, c.Country \
LIMIT 30; ",conn)

,companyName,country,order_count,total_freight
0,Alfreds Futterkiste,Germany,6,225.58
1,Ana Trujillo Emparedados y helados,Mexico,4,97.42
2,Antonio Moreno Taquería,Mexico,7,268.52
3,Around the Horn,UK,13,471.95
4,B's Beverages,UK,10,281.31
5,Berglunds snabbköp,Sweden,18,1559.52
6,Blauer See Delikatessen,Germany,7,168.26
7,Blondesddsl père et fils,France,11,623.66
8,Bon app',France,17,1357.87
9,Bottom-Dollar Markets,Canada,14,793.95


With the INNER JOIN, only customers who have placed at least one order appear in the results, since the join requires matching rows in both tables. The LEFT JOIN includes all customers, even those without orders, showing NULL (or 0 if you wrap with COALESCE) for their freight and order counts. The difference arises because INNER JOIN filters out non-matching rows, while LEFT JOIN preserves them.